# Danh gia truy xuat website voi BM25

In [46]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()
import shutil

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [47]:
import os
import re
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

In [48]:
from nltk.tokenize import sent_tokenize

def split_sentences(text):
    return sent_tokenize(text)

In [87]:
from nltk.tokenize import sent_tokenize
from pyvi import ViTokenizer

def tokenize_vi_sentence_level(text: str) -> list[str]:
    sentences = sent_tokenize(text)
    tokens = []

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        sent_tokens = ViTokenizer.tokenize(sent)
        tokens.extend(sent_tokens.split())

    return tokens


In [88]:
import re

VI_TOKEN_REGEX = re.compile(
    r"[a-zàáạảãâầấậẩẫăằắặẳẵ"
    r"èéẹẻẽêềếệểễ"
    r"ìíịỉĩ"
    r"òóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữ"
    r"ỳýỵỷỹđ0-9_]+$"
)

def is_valid_vi_token(token: str) -> bool:
    return bool(VI_TOKEN_REGEX.fullmatch(token))


In [89]:
def load_stopwords(path):
    with open(path, "r", encoding="utf-8") as f:
        stopwords = set(
            line.strip().lower()
            for line in f
            if line.strip()
        )
    return stopwords

STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

print(f"🛑 Đã load {len(vi_stopwords)} stopword")

🛑 Đã load 1942 stopword


In [90]:
# === Đọc dữ liệu và tiền xử lý ===
import re
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    
    # Xóa URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # text = text.lower()

    # # Loại ký tự không cần thiết (giữ chữ, số, dấu câu cơ bản)
    # text = re.sub(r"[^0-9a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễ"
    #               r"ìíịỉĩòóọỏõôồốộổỗơờớợởỡ"
    #               r"ùúụủũưừứựửữỳýỵỷỹđ\s.,!?]", " ", text)

    # Chuẩn hóa dấu câu
    text = re.sub(r"[.,!?]+", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [91]:
def preprocess_query(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [92]:
STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

query = "Những địa điểm du lịch nổi tiếng nhất ở Hà Nội là gì?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

địa_điểm du_lịch nổi_tiếng hà_nội


In [93]:
import re

def preprocess(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    tokens = [t.lower() for t in tokens if len(t) > 1]
    return " ".join(tokens)


In [94]:
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer

from whoosh.analysis import KeywordAnalyzer

def create_schema():
    # Sử dụng KeywordAnalyzer vì bạn đã preprocess thủ công rồi
    # analyzer này sẽ giữ nguyên các token bạn đã tách
    my_analyzer = KeywordAnalyzer(lowercase=False) 
    return Schema(
        docid=ID(stored=True, unique=True),
        title=TEXT(stored=True, analyzer=my_analyzer),
        content=TEXT(stored=True, analyzer=my_analyzer)
    )


In [95]:
import shutil
import os

def build_index(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = preprocess_query(row["title"])
        doc_file = row["document"]
        # print(f"Indexing docid={docid}, file={doc_file}")
        tokens = doc_terms.get(doc_file, [])
        # print(f"Số token: {len(tokens)}")
        if not tokens:
            continue

        content = preprocess(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [96]:
def readQuery(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query(row["query"])

    return queries


In [97]:
import ast

def readGroundTruth(query_csv, meta_csv):
    meta = pd.read_csv(meta_csv)
    url2docid = dict(zip(meta["url"], meta["id"]))

    df = pd.read_csv(query_csv)
    qrels = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        qrels[qid] = {}

        urls = ast.literal_eval(row["urls"])
        for u in urls:
            if u in url2docid:
                qrels[qid][str(url2docid[u])] = 1

    return qrels


In [98]:
from whoosh.qparser import MultifieldParser
from whoosh.qparser import QueryParser
from whoosh.scoring import BM25F

def bm25_search(ix, query, top_k=100):
    results = {}
    with ix.searcher(weighting=BM25F()) as searcher:
        og = qparser.OrGroup.factory(0.9) # Cho phép OR nhưng ưu tiên các doc chứa nhiều từ hơn
        parser = MultifieldParser(["title", "content"], ix.schema, group=og)
        parser.add_plugin(qparser.FuzzyTermPlugin()) # Có thể thêm tìm kiếm mờ nếu cần
        parser = QueryParser("content", ix.schema, group=qparser.OrGroup)
        q = parser.parse(query)

        hits = searcher.search(q, limit=top_k)

        for hit in hits:
            results[str(hit["docid"])] = float(hit.score)

    return results


In [99]:
def run_bm25_all_queries(ix, queries, top_k=100):
    run = {}

    for qid, query in queries.items():
        run[qid] = bm25_search(ix, query, top_k)

    return run


In [100]:
def evaluate_set_retrieval_at_k(GroundTruth, RunResults, cutoffs=[5,10,20]):
    """
    GroundTruth: dict {qid: {docid: relevance}}
    RunResults : dict {qid: {docid: score}}
    """

    per_query = {}
    avg_metrics = {k: {"P":0, "R":0, "F1":0} for k in cutoffs}
    n = len(GroundTruth)

    print("========== Per-query results ==========")

    for qid in GroundTruth:
        relevant = set(GroundTruth[qid].keys())

        ranked_docs = sorted(
            RunResults.get(qid, {}).items(),
            key=lambda x: x[1],
            reverse=True
        )

        per_query[qid] = {}

        for k in cutoffs:
            retrieved_k = set(docid for docid, _ in ranked_docs[:k])

            tp = len(relevant & retrieved_k)
            fp = len(retrieved_k) - tp
            fn = len(relevant) - tp

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

            per_query[qid][f"P@{k}"] = precision
            per_query[qid][f"R@{k}"] = recall
            per_query[qid][f"F1@{k}"] = f1

            avg_metrics[k]["P"] += precision
            avg_metrics[k]["R"] += recall
            avg_metrics[k]["F1"] += f1

        print(f"Query {qid}")
        for k in cutoffs:
            print(f"  @ {k}")
            print(f"    Precision : {per_query[qid][f'P@{k}']:.4f}")
            print(f"    Recall    : {per_query[qid][f'R@{k}']:.4f}")
            print(f"    F1        : {per_query[qid][f'F1@{k}']:.4f}")
        print("-" * 30)

    print("\n========== Average over all queries ==========")
    for k in cutoffs:
        print(f"@{k}")
        print(f"  Precision : {avg_metrics[k]['P']/n:.4f}")
        print(f"  Recall    : {avg_metrics[k]['R']/n:.4f}")
        print(f"  F1        : {avg_metrics[k]['F1']/n:.4f}")

    return per_query, avg_metrics


## Bo cac tu it xuat hien

In [101]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [102]:
run

{'1': {'494': 11.462062061372126,
  '57': 10.656295321589543,
  '495': 9.472988774345552,
  '172': 9.164560604300666,
  '201': 9.11030517661524,
  '205': 8.980452237083206,
  '207': 8.842663643386814,
  '232': 8.542821857800872,
  '214': 8.51023731410745,
  '216': 8.51023731410745,
  '297': 8.028478579455331,
  '381': 7.971779656973927,
  '405': 7.847748236670792,
  '368': 7.794933459300419,
  '221': 7.515501515911069,
  '224': 6.664298689417482,
  '54': 6.383754205077102,
  '160': 6.201105206948069,
  '313': 6.0353895743190575,
  '67': 5.7947082321204135,
  '391': 5.746873094591473,
  '328': 5.550448927891051,
  '143': 5.548097207922365,
  '410': 5.5343812320886165,
  '162': 5.51462540741702,
  '372': 5.462422222126614,
  '333': 5.421932855487715,
  '349': 5.3570187772461155,
  '400': 5.328704180483119,
  '331': 5.262476352010672,
  '82': 5.23437334928253,
  '363': 5.225478047611589,
  '362': 5.225315077736985,
  '186': 5.20984446568958,
  '150': 5.188964625821146,
  '314': 5.18558421

In [104]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.4000
    Recall    : 0.4444
    F1        : 0.4211
  @ 20
    Precision : 0.2000
    Recall    : 0.4444
    F1        : 0.2759
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.2667
    F1        : 0.4000
  @ 10
    Precision : 0.5000
    Recall    : 0.3333
    F1        : 0.4000
  @

## Khong bo least

In [105]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [106]:
run

{'1': {'494': 11.462062061372126,
  '57': 10.656295321589543,
  '495': 9.472988774345552,
  '172': 9.164560604300666,
  '201': 9.11030517661524,
  '205': 8.980452237083206,
  '207': 8.842663643386814,
  '232': 8.542821857800872,
  '214': 8.51023731410745,
  '216': 8.51023731410745,
  '297': 8.028478579455331,
  '381': 7.971779656973927,
  '405': 7.847748236670792,
  '368': 7.794933459300419,
  '221': 7.515501515911069,
  '224': 6.664298689417482,
  '54': 6.383754205077102,
  '160': 6.201105206948069,
  '313': 6.0353895743190575,
  '67': 5.7947082321204135,
  '391': 5.746873094591473,
  '328': 5.550448927891051,
  '143': 5.548097207922365,
  '410': 5.5343812320886165,
  '162': 5.51462540741702,
  '372': 5.462422222126614,
  '333': 5.421932855487715,
  '349': 5.3570187772461155,
  '400': 5.328704180483119,
  '331': 5.262476352010672,
  '82': 5.23437334928253,
  '363': 5.225478047611589,
  '362': 5.225315077736985,
  '186': 5.20984446568958,
  '150': 5.188964625821146,
  '314': 5.18558421

In [107]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.4000
    Recall    : 0.4444
    F1        : 0.4211
  @ 20
    Precision : 0.2000
    Recall    : 0.4444
    F1        : 0.2759
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.2667
    F1        : 0.4000
  @ 10
    Precision : 0.5000
    Recall    : 0.3333
    F1        : 0.4000
  @

## Them title

In [108]:
import shutil
import os

def build_index_title(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = preprocess_query(row["title"])
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [132]:
from whoosh import qparser
from whoosh.qparser import MultifieldParser
from whoosh.scoring import BM25F

def bm25_search_title(ix, query_str, top_k=100):
    results = {}
    
    # Cấu hình BM25F: b=0.75 là mặc định, có thể tinh chỉnh sau
    # title_B=0.75, content_B=0.75
    weighting = BM25F(B=0.75, K1=1.2)

    with ix.searcher(weighting=weighting) as searcher:
        # 1. Định nghĩa trọng số cho từng trường (Title quan trọng hơn Content)
        # Ở đây ta ưu tiên Title gấp 2 lần Content
        field_boosts = {
            "title": 1.5,
            "content": 1.0
        }

        # 2. Sử dụng OrGroup để tránh việc query quá dài dẫn đến 0 kết quả
        # Những tài liệu chứa nhiều từ khóa hơn vẫn sẽ đứng đầu nhờ thuật toán BM25
        parser = MultifieldParser(
            ["title", "content"],
            schema=ix.schema,
            fieldboosts=field_boosts,
            group=qparser.OrGroup # Chuyển từ AND sang OR
        )

        # Parse câu truy vấn
        q = parser.parse(query_str)
        
        # In ra để debug xem Whoosh thực sự tìm cái gì (Optional)
        # print(f"Query thực tế: {q}")

        hits = searcher.search(q, limit=top_k)

        for hit in hits:
            results[str(hit["docid"])] = float(hit.score)

    return results

In [133]:
def run_bm25_all_queries_title(ix, queries, top_k=50):
    run = {}

    for qid, query in queries.items():
        run[qid] = bm25_search_title(ix, query, top_k)
    return run


In [134]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_title(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries_title(ix, queries, top_k=50)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [129]:
queries

{'1': 'nhà_thờ ở kon_tum',
 '2': 'vũng_tàu có các địa_điểm nào đẹp',
 '3': 'những địa_điểm du_lịch nổi_tiếng nhất ở hà_nội là gì',
 '4': 'nên đi đâu khi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an có những trải nghiệm đặc_sắc nào',
 '6': 'thời_điểm lý_tưởng để du_lịch sa_pa là khi nào',
 '7': 'các điểm tham_quan không nên bỏ lỡ khi đến huế',
 '8': 'du_lịch ninh_bình nên đi tràng_an hay tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật ở miền tây_nam_bộ',
 '10': 'những nơi check in đẹp nhất ở đà_lạt dành cho giới trẻ',
 '11': 'du_lịch hạ_long có những tour và hoạt_động nào hấp_dẫn',
 '12': 'các địa_điểm du_lịch tâm_linh nổi_tiếng ở việt_nam',
 '13': 'nên đi du_lịch côn_đảo vào mùa nào trong năm',
 '14': 'các địa_điểm du_lịch gần tp hcm phù_hợp đi cuối tuần',
 '15': 'du_lịch mộc_châu có gì hấp_dẫn vào mùa hoa',
 '16': 'những vườn quốc_gia đẹp và nổi_tiếng nhất việt_nam',
 '17': 'du_lịch quy_nhơn có những bãi biển hoang_sơ nào',
 '18': 'du_lịch miền huế nên đi đâu',
 '19': 'đồng_nai có nhữn

In [130]:
run

{'1': {'494': 32.04185612397136,
  '57': 25.997820589378627,
  '495': 19.13050270863037,
  '314': 14.317021887732107,
  '362': 13.885029855243493,
  '363': 13.07426816911866,
  '328': 12.92208145614443,
  '464': 12.404074894778322,
  '275': 11.340568722809612,
  '400': 11.05176478825599,
  '298': 9.815557680490233,
  '172': 9.164560604300666,
  '201': 9.11030517661524,
  '205': 8.980452237083206,
  '207': 8.842663643386814,
  '303': 8.687604616681364,
  '232': 8.542821857800872,
  '214': 8.51023731410745,
  '216': 8.51023731410745,
  '297': 8.028478579455331,
  '381': 7.971779656973927,
  '405': 7.847748236670792,
  '284': 7.822286570852348,
  '401': 7.822286570852348,
  '368': 7.794933459300419,
  '221': 7.515501515911069,
  '462': 7.5133310503246475,
  '322': 7.371632528253377,
  '402': 7.371632528253377,
  '453': 6.970075675057791,
  '455': 6.970075675057791,
  '485': 6.970075675057791,
  '488': 6.970075675057791,
  '224': 6.664298689417482,
  '200': 6.61000704463294,
  '54': 6.3837

In [135]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.4000
    Recall    : 0.2500
    F1        : 0.3077
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.3000
    Recall    : 0.3333
    F1        : 0.3158
  @ 20
    Precision : 0.1500
    Recall    : 0.3333
    F1        : 0.2069
------------------------------
Query 4
  @ 5
    Precision : 0.6000
    Recall    : 0.2000
    F1        : 0.3000
  @ 10
    Precision : 0.4000
    Recall    : 0.2667
    F1        : 0.3200
  @

## Tach tu

In [136]:
def preprocess_query_word(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [137]:
import re

def preprocess_word(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    processed = []

    for t in tokens:
        t = t.lower()
        if len(t) <= 1:
            continue

        if "_" in t:
            processed.extend(t.split("_"))
        else:
            processed.append(t)

    return " ".join(processed)


In [138]:
import shutil
import os

def build_index_word(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = preprocess_query_word(row["title"])
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess_word(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [139]:
def readQuery_word(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query(row["query"])

    return queries


In [ ]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_word(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery_word(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = bm25_search_title(ix, queries, top_k=50)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [ ]:
run

In [141]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.2000
    Recall    : 0.1250
    F1        : 0.1538
  @ 10
    Precision : 0.3000
    Recall    : 0.3750
    F1        : 0.3333
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.3000
    Recall    : 0.3333
    F1        : 0.3158
  @ 20
    Precision : 0.1500
    Recall    : 0.3333
    F1        : 0.2069
------------------------------
Query 4
  @ 5
    Precision : 0.6000
    Recall    : 0.2000
    F1        : 0.3000
  @ 10
    Precision : 0.5000
    Recall    : 0.3333
    F1        : 0.4000
  @

## BM25+KNN

In [143]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_title(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries_title(ix, queries, top_k=50)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


### KNN

In [144]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [145]:
def build_tfidf_vectors(meta_csv, json_path):
    import pandas as pd
    import json

    df = pd.read_csv(meta_csv)
    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    docids = []
    corpus = []

    for _, row in df.iterrows():
        docid = str(row["id"])
        doc_file = row["document"]
        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        text = preprocess(tokens)
        docids.append(docid)
        corpus.append(text)

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        lowercase=False,
        norm="l2"
    )

    X = vectorizer.fit_transform(corpus)

    return docids, X, vectorizer


In [146]:
from sklearn.metrics.pairwise import cosine_similarity

def knn_search(query, vectorizer, X_docs, docids, top_k=100):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, X_docs)[0]

    top_idx = np.argsort(sims)[::-1][:top_k]

    results = {}
    for i in top_idx:
        if sims[i] > 0:
            results[docids[i]] = float(sims[i])

    return results


In [147]:
def run_knn_all_queries(queries, vectorizer, X_docs, docids, top_k=100):
    run = {}
    for qid, query in queries.items():
        run[qid] = knn_search(
            query, vectorizer, X_docs, docids, top_k
        )
    return run


In [148]:
def normalize_scores(run):
    norm_run = {}
    for qid, docs in run.items():
        if not docs:
            norm_run[qid] = {}
            continue
        scores = list(docs.values())
        min_s, max_s = min(scores), max(scores)
        norm_run[qid] = {}
        for d, s in docs.items():
            if max_s > min_s:
                norm_run[qid][d] = (s - min_s) / (max_s - min_s)
            else:
                norm_run[qid][d] = 0.0
    return norm_run


In [149]:
def fuse_runs(bm25, knn, alpha=0.6):
    bm25 = normalize_scores(bm25)
    knn = normalize_scores(knn)

    FUSED = {}

    for qid in bm25:
        docs = set(bm25[qid].keys()) | set(knn.get(qid, {}).keys())
        fused_scores = {}
        for d in docs:
            fused_scores[d] = (
                alpha * bm25[qid].get(d, 0) +
                (1 - alpha) * knn.get(qid, {}).get(d, 0)
            )
        FUSED[qid] = fused_scores

    return FUSED


In [157]:
# 1. BM25
RunResults_BM25 = run_bm25_all_queries_title(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn BM25")
# 2. KNN
docids, X_docs, vectorizer = build_tfidf_vectors(META_CSV, JSON_DIR)
RunResults_KNN = run_knn_all_queries(
    queries, vectorizer, X_docs, docids, top_k=100
)
print("✅ Đã chạy xong tất cả truy vấn KNN")
# 3. Fusion
RunResults_Fused = fuse_runs(
    RunResults_BM25,
    RunResults_KNN,
    alpha=0.6
)
print("✅ Đã fuse xong kết quả BM25 và KNN")

✅ Đã chạy xong tất cả truy vấn BM25


c:\Users\mt200\OneDrive\Desktop\AI\InformationRetrieval\Project_InformationRetrieval\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


✅ Đã chạy xong tất cả truy vấn KNN
✅ Đã fuse xong kết quả BM25 và KNN


In [159]:
RunResults_Fused

{'1': {'401': 0.05595733057565158,
  '214': 0.14735469062230575,
  '186': 0.04350649781013662,
  '455': 0.03942256442854344,
  '394': 0.016648310835411045,
  '370': 0.009947699909383708,
  '381': 0.1203433465502804,
  '424': 0.02042680573042312,
  '100': 0.017891293042137442,
  '328': 0.22005619072454724,
  '347': 0.017298397149963365,
  '165': 0.03591814953741308,
  '488': 0.03942256442854344,
  '90': 0.0012431808957677692,
  '172': 0.18143910014709533,
  '131': 0.0003684823003190644,
  '221': 0.11576623595790032,
  '443': 0.03165085244820058,
  '1': 0.024591041944213583,
  '129': 0.03206794739644587,
  '322': 0.047213650760720864,
  '201': 0.18577785512323328,
  '212': 0.0017606734759532854,
  '150': 0.044181389774723384,
  '177': 0.04002125681843911,
  '358': 0.047611383392067615,
  '16': 0.025981513101507113,
  '188': 0.01956898649753113,
  '306': 0.002090858988519681,
  '405': 0.13578845517835897,
  '207': 0.17434830642848897,
  '14': 0.0031078038176537847,
  '109': 0.024809322609

In [158]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    RunResults_Fused
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.3000
    Recall    : 0.3333
    F1        : 0.3158
  @ 20
    Precision : 0.1500
    Recall    : 0.3333
    F1        : 0.2069
------------------------------
Query 4
  @ 5
    Precision : 0.6000
    Recall    : 0.2000
    F1        : 0.3000
  @ 10
    Precision : 0.6000
    Recall    : 0.4000
    F1        : 0.4800
  @

## BM25 + Phan hoi

In [210]:
from whoosh import index, scoring, qparser

from whoosh import index, scoring, qparser
from whoosh.qparser import MultifieldParser

def bm25_retrieve(ix_dir, queries, k1=1.2, b=0.75, top_k=100):
    idx = index.open_dir(ix_dir)

    weighting = scoring.BM25F(
        K1=k1,
        B=b
    )

    searcher = idx.searcher(weighting=weighting)

    parser = MultifieldParser(
        ["title", "content"],
        schema=idx.schema,
        fieldboosts={
            "title": 1.5,
            "content": 1.0
        },
        group=qparser.OrGroup
    )

    runs = {}
    for qid, qtext in queries.items():
        q = parser.parse(qtext)
        results = searcher.search(q, limit=top_k)

        # 🔥 Lưu docnum để dùng cho PRF
        runs[qid] = [hit.docnum for hit in results]

    return runs, searcher



In [211]:
def get_pseudo_relevant(runs, K=10):
    RD = {}
    for qid in runs:
        RD[qid] = runs[qid][:K]
    return RD

In [212]:
from collections import Counter

def collect_term_stats(searcher, RD):
    term_df = Counter()
    R = 0

    for qid in RD:
        for docnum in RD[qid]:
            doc = searcher.stored_fields(docnum)
            if "content" not in doc:
                continue

            terms = set(doc["content"].split())
            for t in terms:
                term_df[t] += 1
            R += 1

    return term_df, R


In [213]:
def estimate_p(term_df, R, K_smooth=0.75, p_prior=0.5):
    p = {}
    for term, df in term_df.items():
        p[term] = (df + K_smooth * p_prior) / (R + K_smooth)
    return p


In [214]:
import math

def compute_idf(searcher, term, field="content"):
    N = searcher.doc_count()
    df = searcher.doc_frequency(field, term)
    return math.log((N - df + 0.5) / (df + 0.5))


In [215]:
import math

def compute_weights(searcher, p_terms):
    weights = {}

    for term, p in p_terms.items():
        if 0 < p < 1:
            idf = compute_idf(searcher, term, field="content")
            weights[term] = idf + math.log(p / (1 - p))

    return weights


In [216]:
def expand_query_safe(original_query, weights, top_m=5, expansion_weight_scale=0.5):
    """
    original_query: Chuỗi văn bản thô (chưa có boost)
    weights: Thống kê trọng số từ PRF
    expansion_weight_scale: Hệ số điều chỉnh độ tin cậy của từ mới (0.1 - 0.5)
    """
    original_terms = original_query.lower().split()
    # Boost từ gốc cố định để giữ đúng ý định ban đầu (Original Intent)
    q_terms = [f"{t}^2.0" for t in original_terms]

    if not weights:
        return " ".join(q_terms)

    # Lọc bỏ các từ đã có trong query gốc trước khi lấy top_m
    filtered_weights = {t: w for t, w in weights.items() if t not in original_terms}
    
    if not filtered_weights:
        return " ".join(q_terms)

    max_w = max(filtered_weights.values())
    sorted_terms = sorted(filtered_weights.items(), key=lambda x: x[1], reverse=True)[:top_m]

    for term, w in sorted_terms:
        if w <= 0: continue
        
        # Boost cho từ mới = (tỉ lệ so với max) * hệ số tin cậy
        # Điều này đảm bảo từ mới không bao giờ quan trọng bằng từ gốc
        boost = (w / max_w) * expansion_weight_scale
        q_terms.append(f"{term}^{round(boost, 2)}")

    return " ".join(q_terms)

In [217]:
from collections import defaultdict

def bm25_prf_iterative(
    ix_dir,
    queries,
    k1=1.2,
    b=0.75,
    K=10,
    top_m=5,
    max_iter=2
):
    current_queries = queries.copy()
    history = []

    for it in range(max_iter):
        runs, searcher = bm25_retrieve(
            ix_dir,
            current_queries,
            k1=k1,
            b=b,
            top_k=100
        )
        print(f"🔥 Iteration {it+1} done.")
        RD = get_pseudo_relevant(runs, K)

        term_df, R = collect_term_stats(searcher, RD)
        p_terms = estimate_p(term_df, R)
        weights = compute_weights(searcher, p_terms)

        new_queries = {}
        for qid, qtext in current_queries.items():
            new_queries[qid] = expand_query_safe(
                qtext,
                weights,
                top_m=top_m,
                expansion_weight_scale=0.4
            )

        history.append({
            "queries": new_queries,
            "weights": weights
        })

        current_queries = new_queries

    return runs, history


In [218]:

# 2. Run PRF on tuning queries
final_run, history = bm25_prf_iterative(
    ix_dir="ind",
    queries=queries,
    k1=1.2,
    b=0.75,
    K=20,
    top_m=5,
    max_iter=2
)

🔥 Iteration 1 done.
🔥 Iteration 2 done.


In [224]:
queries

{'1': 'nhà_thờ ở kon_tum',
 '2': 'vũng_tàu có các địa_điểm nào đẹp',
 '3': 'những địa_điểm du_lịch nổi_tiếng nhất ở hà_nội là gì',
 '4': 'nên đi đâu khi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an có những trải nghiệm đặc_sắc nào',
 '6': 'thời_điểm lý_tưởng để du_lịch sa_pa là khi nào',
 '7': 'các điểm tham_quan không nên bỏ lỡ khi đến huế',
 '8': 'du_lịch ninh_bình nên đi tràng_an hay tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật ở miền tây_nam_bộ',
 '10': 'những nơi check in đẹp nhất ở đà_lạt dành cho giới trẻ',
 '11': 'du_lịch hạ_long có những tour và hoạt_động nào hấp_dẫn',
 '12': 'các địa_điểm du_lịch tâm_linh nổi_tiếng ở việt_nam',
 '13': 'nên đi du_lịch côn_đảo vào mùa nào trong năm',
 '14': 'các địa_điểm du_lịch gần tp hcm phù_hợp đi cuối tuần',
 '15': 'du_lịch mộc_châu có gì hấp_dẫn vào mùa hoa',
 '16': 'những vườn quốc_gia đẹp và nổi_tiếng nhất việt_nam',
 '17': 'du_lịch quy_nhơn có những bãi biển hoang_sơ nào',
 '18': 'du_lịch miền huế nên đi đâu',
 '19': 'đồng_nai có nhữn

In [219]:
final_run

{'1': [437,
  57,
  438,
  244,
  259,
  250,
  272,
  313,
  314,
  285,
  409,
  349,
  171,
  196,
  200,
  201,
  221,
  206,
  207,
  258,
  331,
  354,
  319,
  212,
  264,
  214,
  54,
  159,
  271,
  407,
  350,
  66,
  340,
  142,
  359,
  279,
  351,
  161,
  323,
  289,
  304,
  288,
  81,
  398,
  400,
  428,
  431,
  183,
  149,
  341,
  339,
  195,
  80,
  391,
  77,
  166,
  74,
  176,
  310,
  164,
  223,
  389,
  369,
  370,
  399,
  5,
  225,
  33,
  128,
  198,
  83,
  112,
  1,
  16,
  302,
  343,
  94,
  185,
  108,
  346,
  238,
  324,
  61,
  99,
  311,
  133,
  321,
  11,
  240,
  150,
  189,
  37,
  276,
  113,
  266,
  79,
  41,
  91,
  38,
  123],
 '2': [280,
  204,
  421,
  371,
  122,
  373,
  9,
  279,
  205,
  250,
  211,
  370,
  219,
  259,
  324,
  244,
  415,
  331,
  286,
  408,
  283,
  420,
  369,
  332,
  366,
  343,
  317,
  311,
  130,
  393,
  287,
  436,
  435,
  434,
  170,
  141,
  433,
  298,
  194,
  392,
  347,
  368,
  372,
  431,
  321,

In [220]:
from whoosh import index

def runs_docnum_to_eval_format(ix_dir, runs):
    """
    runs: {qid: [docnum, docnum, ...]}
    return: {qid: {docid: score}}
    """
    idx = index.open_dir(ix_dir)
    RunResults = {}

    with idx.searcher() as searcher:
        for qid, docnums in runs.items():
            scores = {}
            n = len(docnums)

            for rank, docnum in enumerate(docnums):
                doc = searcher.stored_fields(docnum)
                docid = str(doc["docid"])

                # score giả theo rank (cao → tốt)
                scores[docid] = float(n - rank)

            RunResults[qid] = scores

    return RunResults


In [206]:
RunResults_for_eval = runs_docnum_to_eval_format(
    ix_dir="ind",
    runs=final_run   # hoặc runs hiện tại của bạn
)

In [223]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    RunResults_for_eval
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 2
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 3
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 4
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @

## BM25 + KNN + PageRank